In [ ]:
import sys
import os
import setuptools.dist
import tensorflow as tf
import tensorflow_probability as tfp
import numpy as np
from matplotlib import pyplot as plt

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))
try:
    from config import DATA_PATH, HMSC_HPC_PATH
except ImportError:
    DATA_PATH = os.getenv('GLC_DATA_PATH')
    HMSC_HPC_PATH = os.getenv('HMSC_HPC_PATH')

sys.path.append(HMSC_HPC_PATH)

path_data = DATA_PATH
modelTypeString = "nc0071_ns1000_np0000_nf00"
samN = 100
thinN = 10
nChains = 1
eagerExecFlag = 0
fp = 64
RS = 1
transient = samN*thinN

In [ ]:
verbose = np.maximum(1, int((transient+samN*thinN)/100))
dtype = np.float32 if fp == 32 else np.float64
tf.config.run_functions_eagerly(eagerExecFlag)
input_path = os.path.join(path_data, "hmsc", "init", "init_%s_chain01.rds" % modelTypeString)
output_path = os.path.join(path_data, "hmsc", f"fmTF_localhostname{fp}", "TF_%s_chain%.2d_sam%.4d_thin%.4d.rds" % (modelTypeString, nChains, samN, thinN))
os.makedirs(os.path.join(path_data, "hmsc", f"fmTF_localhostname{fp}"), exist_ok=True)
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    tf.config.experimental.set_virtual_device_configuration(gpus[0], [tf.config.experimental.VirtualDeviceConfiguration(memory_limit=11400)])

# gpus = tf.config.list_physical_devices('GPU')
# tf.config.experimental.set_memory_growth(gpus[0], True)

args="\"\"--samples %d --transient %d --thin %d --verbose %d --input %s --output %s --fse 1 --tnlib tf --profile 0 --fp %d --eager %d\"\"" % (samN, transient, thinN, verbose, input_path, output_path, fp, 0)
run_cmd = '\"' + os.path.join(HMSC_HPC_PATH, 'hmsc/run_gibbs_sampler.py') + '\"' + \" \" + args
%run $run_cmd

### Local

ns 1000, 100000 iterations, 14164s

### Mahti

ns 1000, 2000 iterations, 32.5s

ns 2519, 2000 iterations, 74s; 200000 iterations, 7440s

### Spatial 100, Mahti

ns 2519, 2000 iterations, 135s